## 🎯 Common PySpark Interview Questions - Nested JSON Processing

### Q1: What's the difference between explode() and explode_outer()?
**Answer:**
- `explode()`: Creates a row for each element in array/map. **Drops rows if array is NULL or empty**
- `explode_outer()`: Same as explode but **preserves rows with NULL/empty arrays** (fills with NULL)
- **Production Best Practice**: Always use `explode_outer()` to avoid silent data loss

**Example:**
```python
# Input: [{id: 1, items: [a, b]}, {id: 2, items: []}, {id: 3, items: null}]

df.select(col("id"), explode(col("items")))  
# Output: [(1, a), (1, b)]  <- Rows 2 and 3 are LOST!

df.select(col("id"), explode_outer(col("items")))
# Output: [(1, a), (1, b), (2, null), (3, null)]  <- All rows preserved
```

---

### Q2: Why use explicit schema instead of schema inference?
**Answer:**

**Schema Inference Problems:**
1. **Performance**: Scans entire dataset to infer types (double read)
2. **Inconsistency**: May infer wrong types from sample data
3. **Production Risk**: Schema can change between runs
4. **Scalability**: Expensive on large files (TBs)

**Explicit Schema Benefits:**
1. ✅ Single data scan (faster)
2. ✅ Type safety guaranteed
3. ✅ Schema version control
4. ✅ Better for nested structures

**Interview Tip**: Always mention performance impact in production scenarios

---

### Q3: How do you handle deeply nested JSON (array within array)?
**Answer:**

**Strategy: Sequential Explosion**
1. Explode outer array first
2. Flatten the struct fields from exploded array
3. Explode inner nested arrays
4. Repeat until fully flattened

```python
# Level 1: Explode items array
df1 = df.select("*", explode_outer("items").alias("item"))

# Level 2: Flatten item struct and keep nested offers
df2 = df1.select("*", "item.offers").drop("items", "item")

# Level 3: Explode nested offers array
df3 = df2.select("*", explode_outer("offers").alias("offer"))
```

**Key Point**: Process one level at a time, always flatten structs between explosions

---

### Q4: How to optimize nested JSON processing for large datasets?
**Answer:**

**Optimization Techniques:**

1. **Schema Definition**: Use explicit schema (avoid inference)
2. **Early Filtering**: Apply filters before explosion
   ```python
   df.filter("date >= '2026-01-01'").select(...).explode(...)
   ```
3. **Column Pruning**: Select only needed columns before explosion
4. **Partition Pruning**: Partition Delta tables by date/region
5. **Broadcast Joins**: For small dimension tables post-explosion
6. **Repartition**: After explosion if data skew occurs
   ```python
   df_exploded.repartition(200, "customer_id")
   ```
7. **Cache Strategically**: Only if reusing exploded DF multiple times
8. **Delta Optimization**: Use `OPTIMIZE` and `ZORDER BY` on final tables

---

### Q5: What's the multiLine option in JSON reading?
**Answer:**

**multiLine=False (default)**:
- Each line is a complete JSON object
- Faster and more parallelizable
- **Format**: JSONL (JSON Lines)
```json
{"id": 1, "name": "John"}
{"id": 2, "name": "Jane"}
```

**multiLine=True**:
- Entire file is one JSON array or object spans multiple lines
- Slower (less parallelization)
- **Format**: Pretty-printed JSON
```json
{
  "id": 1,
  "name": "John"
}
```

**Production Tip**: Use `multiLine=false` with JSONL format for better performance

---

### Q6: How to handle missing/null fields in nested JSON?
**Answer:**

**Techniques:**

1. **Schema with nullable=True**
   ```python
   StructField("phone", StringType(), True)  # Allows NULL
   ```

2. **coalesce() for default values**
   ```python
   .withColumn("phone", coalesce(col("phone"), lit("UNKNOWN")))
   ```

3. **explode_outer() for arrays**
   ```python
   explode_outer(col("items"))  # Keeps rows with NULL arrays
   ```

4. **when().otherwise() for conditional logic**
   ```python
   .withColumn("discount", 
       when(col("offers").isNotNull(), col("offer_value"))
       .otherwise(lit(0)))
   ```

---

### Q7: Explain StructType vs ArrayType vs MapType
**Answer:**

| Type | Represents | Example | Access Method |
|------|------------|---------|---------------|
| **StructType** | Nested object/struct with named fields | `{"name": "John", "age": 30}` | `df.col.name` |
| **ArrayType** | List/array of same type elements | `[1, 2, 3, 4]` | `explode()`, `col[0]` |
| **MapType** | Key-value pairs (dictionary) | `{"key1": "val1", "key2": "val2"}` | `map_keys()`, `map_values()` |

**Nested Example:**
```python
# Array of Structs
ArrayType(StructType([
    StructField("product", StringType()),
    StructField("price", DoubleType())
]))

# Struct containing Array
StructType([
    StructField("customer_id", StringType()),
    StructField("orders", ArrayType(StringType()))
])
```

---

### Q8: How to flatten all nested columns automatically?
**Answer:**

**Challenge**: Manually flattening is tedious for deeply nested schemas

**Solution**: Recursive function to flatten all levels

```python
def flatten_df(df):
    # Get all fields from dataframe
    flat_cols = []
    nested_cols = []
    
    for field in df.schema.fields:
        if isinstance(field.dataType, StructType):
            nested_cols.append(field.name)
        else:
            flat_cols.append(col(field.name))
    
    # Expand nested columns
    for nested_col in nested_cols:
        for field in df.schema[nested_col].dataType.fields:
            flat_cols.append(
                col(f"{nested_col}.{field.name}").alias(f"{nested_col}_{field.name}")
            )
    
    df_flat = df.select(flat_cols)
    
    # Recursively flatten if still nested
    if nested_cols:
        return flatten_df(df_flat)
    return df_flat

# Usage
df_flattened = flatten_df(df_nested)
```

**Interview Tip**: Mention this shows understanding of recursion and schema introspection

---

### Q9: What's the impact of partition strategy on nested data?
**Answer:**

**Good Partitioning Choices:**
- Date columns: `order_year`, `order_month`, `order_date`
- Low-cardinality categorical: `region`, `status`, `category`
- Enables partition pruning (huge performance gain)

**Bad Partitioning Choices:**
- High cardinality: `customer_id`, `order_id` (creates too many small files)
- After explosion: Partitioning on exploded fields can cause data skew

**Best Practice:**
```python
# Partition by time-based columns
df.write.partitionBy("order_year", "order_month").delta(path)

# Query benefits from partition pruning
df.filter("order_year = 2026 AND order_month = 5")  # Only reads 1 partition!
```

**Rule of Thumb**: Aim for 100MB-1GB per partition, avoid >10,000 partitions

---

### Q10: How to handle schema evolution in nested JSON?
**Answer:**

**Challenge**: New fields added to production JSON over time

**Solutions:**

1. **Schema Merging (Delta Lake)**
   ```python
   df.write.format("delta") \
       .mode("append") \
       .option("mergeSchema", "true") \
       .save(path)
   ```

2. **Explicit Schema with nullable=True**
   - All fields nullable to handle missing data
   - Add new fields to schema before they appear

3. **Schema Registry (Kafka/Confluent)**
   - Centralized schema management
   - Version control for schema changes

4. **Try-Except Pattern**
   ```python
   try:
       df = spark.read.schema(latest_schema).json(path)
   except:
       df = spark.read.schema(fallback_schema).json(path)
   ```

**Production Pattern**: Use Delta Lake with `mergeSchema=true` for automatic evolution

## 🎓 Key Takeaways - Production Best Practices

### ✅ Schema Management
1. **Always define explicit schema** for production workloads
2. Use `StructType`, `ArrayType`, `StructField` with proper nullable flags
3. Avoid schema inference on large datasets (performance killer)

### ✅ Array Handling
1. **Use `explode_outer()`** instead of `explode()` to prevent data loss
2. Process nested arrays sequentially (one level at a time)
3. Flatten structs between array explosions

### ✅ Performance Optimization
1. Apply filters **before** explosions (predicate pushdown)
2. Select only needed columns early (column pruning)
3. Partition Delta tables by low-cardinality, frequently-filtered columns
4. Use `OPTIMIZE` and `ZORDER BY` on Delta tables
5. Broadcast small dimension tables in joins

### ✅ Data Quality
1. Add validation checks before writing to Silver/Gold layers
2. Handle NULL/missing values explicitly with `coalesce()` or `when()`
3. Add audit columns: `ingestion_timestamp`, `source_system`
4. Filter invalid records and log them separately

### ✅ Medallion Architecture
- **Bronze**: Raw JSON as-is (minimal transformation)
- **Silver**: Cleaned, flattened, validated, business keys added
- **Gold**: Aggregated metrics, business logic applied

### ✅ Delta Lake Features
1. ACID transactions guarantee consistency
2. Time travel for auditing and rollback
3. Schema evolution with `mergeSchema=true`
4. Efficient upserts with `MERGE INTO`
5. Optimize storage with `OPTIMIZE` and `ZORDER BY`

---

## 📚 Additional Resources

### Official Documentation
- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
- [Delta Lake Guide](https://docs.delta.io/latest/index.html)
- [Databricks Best Practices](https://docs.databricks.com/delta/best-practices.html)

### Interview Preparation
- Focus on **explode vs explode_outer** (most common mistake)
- Understand **schema definition** deeply (StructType composition)
- Know **performance optimization** techniques
- Practice **nested data modeling** problems
- Understand **Medallion Architecture** pattern

---

## 🎯 Interview Success Tips

1. **Always mention production considerations**
   - Don't just solve the problem, explain why your solution is production-ready

2. **Discuss trade-offs**
   - "We could use explode() for simplicity, but explode_outer() is safer in production"

3. **Performance awareness**
   - Mention partitioning, caching, broadcast joins proactively

4. **Data quality focus**
   - Show you think about NULL handling, validation, and edge cases

5. **Real-world context**
   - Reference Medallion Architecture, Delta Lake, Azure/AWS storage

---

**🎉 You're now ready for production PySpark nested JSON processing!**

In [0]:
from pyspark.sql.functions import sum, avg, count, col, desc

print("📊 BUSINESS ANALYTICS QUERIES")
print("=" * 80)

# 1. Revenue by Category
print("\n1️⃣ Revenue by Product Category:")
revenue_by_category = df_silver.groupBy("category") \
    .agg(
        sum("final_item_total").alias("total_revenue"),
        count("order_id").alias("order_count"),
        avg("final_item_total").alias("avg_order_value")
    ) \
    .orderBy(desc("total_revenue"))
display(revenue_by_category)

# 2. Top Customers by Spend
print("\n2️⃣ Top Customers by Total Spend:")
top_customers = df_silver.groupBy("customer_id", "customer_name", "city", "state") \
    .agg(
        sum("final_item_total").alias("total_spend"),
        count("order_id").alias("order_count")
    ) \
    .orderBy(desc("total_spend"))
display(top_customers)

# 3. Payment Mode Analysis
print("\n3️⃣ Revenue by Payment Mode:")
payment_analysis = df_silver.groupBy("payment_mode") \
    .agg(
        sum("final_item_total").alias("total_revenue"),
        count("*").alias("transaction_count"),
        avg("final_item_total").alias("avg_transaction_value")
    ) \
    .orderBy(desc("total_revenue"))
display(payment_analysis)

# 4. Discount Impact Analysis
print("\n4️⃣ Discount Impact Analysis:")
discount_analysis = df_silver.groupBy("offer_type") \
    .agg(
        count("*").alias("items_with_offer"),
        sum("offer_discount_amount").alias("total_discount_given"),
        avg("offer_discount_amount").alias("avg_discount_per_item")
    ) \
    .orderBy(desc("total_discount_given"))
display(discount_analysis)

# 5. Geographic Revenue Distribution
print("\n5️⃣ Revenue by State:")
state_revenue = df_silver.groupBy("state", "city") \
    .agg(
        sum("final_item_total").alias("total_revenue"),
        count("order_id").alias("order_count")
    ) \
    .orderBy(desc("total_revenue"))
display(state_revenue)

## 🚀 Performance Optimization Best Practices

### 1. Schema Definition
✅ **Use explicit schema** instead of schema inference  
❌ Bad: `spark.read.json(path)` - scans all data twice  
✅ Good: `spark.read.schema(my_schema).json(path)` - reads once

### 2. Array Explosion
✅ **Use explode_outer()** for nullable arrays  
❌ Bad: `explode()` - drops NULL/empty arrays silently  
✅ Good: `explode_outer()` - preserves all records

### 3. Column Projection
✅ **Select only needed columns early** in transformation  
❌ Bad: `df.select("*").filter(...).select(cols)`  
✅ Good: `df.select(cols).filter(...)`

### 4. Predicate Pushdown
✅ **Filter data as early as possible**  
❌ Bad: Read all data → filter → process  
✅ Good: `df.filter("date >= '2026-01-01'")` early

### 5. Partitioning Strategy
✅ **Partition by frequently filtered columns**  
✅ Good partitions: date, region, status  
❌ Bad partitions: customer_id (high cardinality)

### 6. File Optimization
✅ **Use Delta Lake with OPTIMIZE and Z-ORDER**  
```python
OPTIMIZE delta_table ZORDER BY (order_date, customer_id)
```

### 7. Broadcast Joins
✅ **Broadcast small dimension tables**  
```python
from pyspark.sql.functions import broadcast
df.join(broadcast(small_df), "key")
```

### 8. Caching Strategy
✅ **Cache only when reusing DataFrames multiple times**  
```python
df.cache()  # Only if used 3+ times
```

In [0]:
# Write to spark-warehouse (works with Delta partitioning)
# In production, this would be on external storage (S3/ADLS/GCS)
delta_table_name = "orders_detailed_silver"

# Write to Delta Lake with partitioning
df_quality_checked.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .option("overwriteSchema", "true") \
    .saveAsTable(delta_table_name)

print("✅ Data successfully written to Delta Lake!")
print(f"\n💾 Delta Table: {delta_table_name}")
print("\n📁 Partition Structure: order_year/order_month")
print("\n🔍 Verify written data:")

# Read back and verify
df_silver = spark.table(delta_table_name)
print(f"\n📊 Total records in Silver layer: {df_silver.count()}")
print(f"📊 Total columns: {len(df_silver.columns)}")

display(df_silver.select(
    "order_id", "order_date", "customer_name", "city", "state",
    "product_name", "category", "quantity", "unit_price",
    "final_item_total", "payment_mode", "order_status"
).orderBy("order_date", "order_id"))

## Step 10: Verify Delta Table and Run Analysis

In [0]:
# Display Delta table metadata
print("📋 DELTA TABLE METADATA")
print("=" * 80)

# Show partitions
print("\n📂 Partition Structure:")
dbutils.fs.ls(delta_path)

# Display schema
print("\n📋 Final Silver Layer Schema:")
df_silver.printSchema()

# Show table statistics
print("\n📊 Data Statistics:")
df_silver.select(
    "order_id", "customer_id", "product_name", 
    "quantity", "final_item_total"
).describe().show()

# Count by partition
print("\n📊 Records by Partition:")
display(df_silver.groupBy("order_year", "order_month").count().orderBy("order_year", "order_month"))

## Step 11: Sample Business Analytics Queries

Now that data is in Delta, run typical business queries

In [0]:
from pyspark.sql.functions import col

# Count before quality checks
count_before = df_with_calculations.count()

# Apply data quality filters
df_quality_checked = df_with_calculations.filter(
    # Primary key must not be NULL
    col("order_id").isNotNull() &
    
    # Customer ID must exist
    col("customer_id").isNotNull() &
    
    # Quantity and price validations
    (col("quantity") > 0) &
    (col("unit_price") >= 0) &
    
    # Final total should be non-negative
    (col("final_item_total") >= 0) &
    
    # Valid order status
    col("order_status").isin(["PENDING", "PROCESSING", "SHIPPED", "COMPLETED", "CANCELLED"])
)

# Count after quality checks
count_after = df_quality_checked.count()
count_filtered = count_before - count_after

print("✅ Data quality validations applied!")
print("\n📊 Quality Check Summary:")
print(f"  Records before validation: {count_before}")
print(f"  Records after validation: {count_after}")
print(f"  Records filtered out: {count_filtered}")
print(f"  Data quality pass rate: {(count_after/count_before)*100:.2f}%")

print("\n✅ Applied Validations:")
print("  • order_id IS NOT NULL")
print("  • customer_id IS NOT NULL")
print("  • quantity > 0")
print("  • unit_price >= 0")
print("  • final_item_total >= 0")
print("  • order_status IN valid values")

print("\n🔍 Clean Data Sample:")
display(df_quality_checked.limit(10))

## Step 9: Write to Delta Lake (Silver Layer)

**Delta Lake Features Used**:
- ✅ **Partitioning**: By order_year and order_month for query pruning
- ✅ **Overwrite Mode**: Replace existing data
- ✅ **Optimized Write**: Better file sizes
- ✅ **ACID Transactions**: Guaranteed consistency

**Medallion Architecture**:
- **Bronze**: Raw JSON (already in /tmp/nested_json_demo/)
- **Silver**: Cleaned, flattened, validated data ← *We are here*
- **Gold**: Aggregated business metrics

**Production Path**: Would be `/mnt/silver/orders/` in ADLS/S3

In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, year, month, round, 
    when, coalesce, lit
)

# Add calculated columns
df_with_calculations = df_final_exploded \
    .withColumn(
        "item_subtotal",
        round(col("quantity") * col("unit_price"), 2)
    ) \
    .withColumn(
        "line_discount_amount",
        round((col("item_subtotal") * col("discount_percent")) / 100, 2)
    ) \
    .withColumn(
        "offer_discount_amount",
        round(coalesce(col("offer_discount_value"), lit(0.0)), 2)
    ) \
    .withColumn(
        "final_item_total",
        round(
            col("item_subtotal") - col("line_discount_amount") - col("offer_discount_amount"),
            2
        )
    ) \
    .withColumn(
        "total_discount",
        round(col("line_discount_amount") + col("offer_discount_amount"), 2)
    ) \
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    ) \
    .withColumn(
        "order_year",
        year(col("order_date"))
    ) \
    .withColumn(
        "order_month",
        month(col("order_date"))
    )

print("✅ Derived columns added successfully!")
print("\n📊 New Calculated Fields:")
print("  • item_subtotal = quantity * unit_price")
print("  • line_discount_amount = (subtotal * discount_percent) / 100")
print("  • offer_discount_amount = offer discount value (or 0 if NULL)")
print("  • final_item_total = subtotal - line_discount - offer_discount")
print("  • total_discount = line_discount + offer_discount")
print("  • ingestion_timestamp = current timestamp (audit trail)")
print("  • order_year, order_month = partitioning columns")

print("\n🔍 Sample with Calculations:")
display(df_with_calculations.select(
    "order_id", "product_name", "quantity", "unit_price",
    "item_subtotal", "discount_percent", "line_discount_amount",
    "offer_discount_value", "final_item_total", "total_discount"
).orderBy("order_id"))

## Step 8: Data Quality Validations

**Critical Quality Checks**:
1. Remove records with NULL order_id (primary key)
2. Filter negative quantities or prices
3. Validate payment status
4. Check for valid customer information

**Production Pattern**: Always validate data before writing to Silver/Gold layers

In [0]:
# Explode nested offers array (array within array scenario)
df_offers_exploded = df_items_flat.select(
    col("*"),
    explode_outer(col("offers")).alias("offer")  # explode_outer handles NULL/empty arrays
).drop("offers")  # Drop original offers array

# Flatten offer struct fields
df_final_exploded = df_offers_exploded.select(
    col("*"),
    col("offer.offer_id").alias("offer_id"),
    col("offer.offer_type").alias("offer_type"),
    col("offer.discount_value").alias("offer_discount_value")
).drop("offer")  # Drop offer struct

print("✅ Nested offers array exploded successfully!")
print("\n📊 Explosion Summary:")
print(f"  Original orders: 3")
print(f"  After items explosion: {df_items_flat.count()} rows")
print(f"  After offers explosion: {df_final_exploded.count()} rows")
print("\n💡 Note: Rows with empty offers array are preserved (NULL offer fields)")
print("\n🔍 Sample Data with Exploded Offers:")
display(df_final_exploded.select(
    "order_id", "product_name", "quantity", "unit_price", 
    "discount_percent", "offer_id", "offer_type", "offer_discount_value"
).orderBy("order_id", "product_name"))

## Step 7: Add Calculated/Derived Columns

**Business Logic**:
1. Calculate line-level discount amount
2. Calculate offer discount amount
3. Calculate final item total after all discounts
4. Add ingestion timestamp for audit trail
5. Add year/month partitions for query optimization

**Production Pattern**: Add business metrics at transformation time

In [0]:
from pyspark.sql.functions import explode_outer, col

# Explode items array to get one row per item
df_items_exploded = df_flattened.select(
    col("*"),
    explode_outer(col("items")).alias("item")  # Use explode_outer to handle empty/null arrays
).drop("items")  # Drop original array column

# Flatten item struct fields
df_items_flat = df_items_exploded.select(
    col("*"),
    col("item.item_id").alias("item_id"),
    col("item.product_name").alias("product_name"),
    col("item.category").alias("category"),
    col("item.quantity").alias("quantity"),
    col("item.unit_price").alias("unit_price"),
    col("item.discount_percent").alias("discount_percent"),
    col("item.offers").alias("offers")  # Keep offers array for next explosion
).drop("item")  # Drop item struct

print("✅ Items array exploded successfully!")
print(f"📊 Rows after items explosion: {df_items_flat.count()}")
print("\n📊 Row Count Comparison:")
print(f"  Before explosion: {df_flattened.count()} orders")
print(f"  After explosion: {df_items_flat.count()} order-item combinations")
print("\n🔍 Sample Data After Items Explosion:")
display(df_items_flat.select(
    "order_id", "customer_name", "city", "item_id", 
    "product_name", "quantity", "unit_price", "offers"
))

## Step 6: Explode Nested Offers Array (Array within Array)

**Challenge**: Items array contains offers array - need double explosion

**Real-world scenario**: 
- Order has multiple items
- Each item can have multiple promotional offers
- Need to create one row per offer for detailed analysis

**Using explode_outer()**: Critical here because some items have NO offers

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date

# Step 1: Flatten nested customer struct (including nested address)
df_flattened = df_raw.select(
    # Order level fields
    col("order_id"),
    to_date(col("order_date")).alias("order_date"),  # Convert to date type
    col("order_timestamp"),
    col("order_status"),
    
    # Flatten customer struct (Level 1 nesting)
    col("customer.customer_id").alias("customer_id"),
    col("customer.customer_name").alias("customer_name"),
    col("customer.email").alias("customer_email"),
    col("customer.phone").alias("customer_phone"),
    
    # Flatten customer.address struct (Level 2 nesting)
    col("customer.address.street").alias("street"),
    col("customer.address.city").alias("city"),
    col("customer.address.state").alias("state"),
    col("customer.address.zipcode").alias("zipcode"),
    col("customer.address.country").alias("country"),
    
    # Flatten payment struct
    col("payment.payment_id").alias("payment_id"),
    col("payment.payment_mode").alias("payment_mode"),
    col("payment.payment_status").alias("payment_status"),
    col("payment.transaction_id").alias("transaction_id"),
    
    # Keep items array as-is (will explode next)
    col("items")
)

print("✅ Nested structs flattened successfully!")
print(f"📊 Columns after flattening: {len(df_flattened.columns)}")
print("\n📋 Flattened Schema:")
df_flattened.printSchema()

print("\n🔍 Sample Flattened Data:")
display(df_flattened)

## Step 5: Explode Items Array

**explode() vs explode_outer()**:
- `explode()`: Drops rows if array is NULL or empty ❌
- `explode_outer()`: Keeps rows even if array is NULL/empty ✅

**Production Best Practice**: Always use `explode_outer()` to avoid silent data loss

**Effect**: Converts array to rows (one row per array element)

In [0]:
# Read JSON data with explicit schema
workspace_json_path = "/Workspace/Users/pavankumarreddy3247@gmail.com/Retail Sales/retail-medallion-project/nested_json_demo/orders.json"

df_raw = spark.read \
    .option("multiLine", "false") \
    .schema(order_schema) \
    .json(workspace_json_path)

print("✅ Data loaded successfully!")
print(f"📊 Total Records: {df_raw.count()}")
print(f"📊 Total Columns: {len(df_raw.columns)}")
print("\n📋 Complete Schema Tree:")
df_raw.printSchema()

# Display sample data
print("\n🔍 Sample Raw Data:")
display(df_raw)

## Step 4: Flatten Nested Struct Columns

**Strategy**: Use dot notation to access nested fields
- `customer.customer_id` extracts field from nested struct
- `customer.address.city` extracts from doubly-nested struct
- Flattening improves query performance and readability

**Production Pattern**: Create a wide, denormalized view for analytics

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, ArrayType, TimestampType, DateType
)

# Define schema for deeply nested JSON
# Level 4: Offers schema (nested inside items array)
offer_schema = StructType([
    StructField("offer_id", StringType(), True),
    StructField("offer_type", StringType(), True),
    StructField("discount_value", DoubleType(), True)
])

# Level 3: Items schema (array of items, each containing array of offers)
item_schema = StructType([
    StructField("item_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_percent", IntegerType(), True),
    StructField("offers", ArrayType(offer_schema), True)  # Array inside array
])

# Level 2: Nested address schema (struct inside customer struct)
address_schema = StructType([
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("country", StringType(), True)
])

# Level 2: Customer schema (contains nested address)
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("address", address_schema, True)  # Nested struct
])

# Level 2: Payment schema
payment_schema = StructType([
    StructField("payment_id", StringType(), True),
    StructField("payment_mode", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("transaction_id", StringType(), True)
])

# Level 1: Root order schema (main structure)
order_schema = StructType([
    StructField("order_id", StringType(), False),  # NOT NULL
    StructField("order_date", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("customer", customer_schema, True),  # Nested struct
    StructField("items", ArrayType(item_schema), True),  # Array of structs
    StructField("payment", payment_schema, True),  # Nested struct
    StructField("order_status", StringType(), True)
])

print("✅ Schema defined successfully!")
print("\n📋 Schema Structure:")
print("\nRoot Level:")
for field in order_schema.fields:
    if isinstance(field.dataType, StructType):
        print(f"  ├─ {field.name}: STRUCT (nested {len(field.dataType.fields)} fields)")
    elif isinstance(field.dataType, ArrayType):
        print(f"  ├─ {field.name}: ARRAY")
    else:
        print(f"  ├─ {field.name}: {field.dataType}")

print("\n💡 Key Points:")
print("  • order_id is NOT NULL (nullable=False)")
print("  • customer contains nested address struct")
print("  • items is ArrayType containing StructType")
print("  • Each item contains nested offers array (array in array)")
print("  • payment is a nested struct")

## Step 3: Read Multiline JSON Data

**Key Options**:
- `multiLine=True`: Each JSON object spans multiple lines (pretty-printed)
- `schema=order_schema`: Use explicit schema (no inference)
- `mode="PERMISSIVE"`: Handle corrupt records gracefully

**Production Best Practice**: Always use explicit schema in production to avoid schema inference overhead on every read.

# Processing Deeply Nested JSON with PySpark - Production Guide

## Scenario
Real-time e-commerce analytics platform processing order transactions with:
- **Nested customer details** (address struct)
- **Items array** with nested offers array
- **Payment information** (nested struct)
- **Data quality validations**
- **Medallion Architecture** (Bronze → Silver layer)

## Technologies
- PySpark 3.x
- Databricks Runtime
- Delta Lake
- Azure Data Lake Storage
- Unity Catalog

## Key Learning Points
✅ Explicit schema definition with StructType/ArrayType  
✅ Multiline JSON handling  
✅ Flattening nested structs  
✅ Exploding arrays (explode vs explode_outer)  
✅ Derived column calculations  
✅ Data quality validations  
✅ Performance optimization  
✅ Delta Lake partitioning  

## Step 1: Create Sample Nested JSON Data

We'll create realistic e-commerce order data with:
- **Level 1**: Order and Customer info
- **Level 2**: Nested customer address (struct)
- **Level 3**: Items array
- **Level 4**: Nested offers array inside each item
- **Level 2**: Payment struct

This represents a typical microservices payload from an order management system.

In [0]:
# Create sample nested JSON data representing e-commerce orders
import json
from datetime import datetime, timedelta
import random

# Sample nested JSON with multiple complexity levels
sample_orders = [
    {
        "order_id": "ORD-2026-001",
        "order_date": "2026-05-15",
        "order_timestamp": "2026-05-15T10:30:45Z",
        "customer": {
            "customer_id": "CUST-1001",
            "customer_name": "John Doe",
            "email": "john.doe@email.com",
            "phone": "+1-555-0101",
            "address": {
                "street": "123 Main Street",
                "city": "Seattle",
                "state": "WA",
                "zipcode": "98101",
                "country": "USA"
            }
        },
        "items": [
            {
                "item_id": "ITM-001",
                "product_name": "Wireless Headphones",
                "category": "Electronics",
                "quantity": 2,
                "unit_price": 79.99,
                "discount_percent": 10,
                "offers": [
                    {"offer_id": "OFF-001", "offer_type": "SEASONAL", "discount_value": 5.0},
                    {"offer_id": "OFF-002", "offer_type": "LOYALTY", "discount_value": 3.0}
                ]
            },
            {
                "item_id": "ITM-002",
                "product_name": "USB-C Cable",
                "category": "Accessories",
                "quantity": 3,
                "unit_price": 12.99,
                "discount_percent": 0,
                "offers": []
            }
        ],
        "payment": {
            "payment_id": "PAY-5001",
            "payment_mode": "Credit Card",
            "payment_status": "SUCCESS",
            "transaction_id": "TXN-98765"
        },
        "order_status": "COMPLETED"
    },
    {
        "order_id": "ORD-2026-002",
        "order_date": "2026-05-16",
        "order_timestamp": "2026-05-16T14:22:18Z",
        "customer": {
            "customer_id": "CUST-1002",
            "customer_name": "Jane Smith",
            "email": "jane.smith@email.com",
            "phone": "+1-555-0202",
            "address": {
                "street": "456 Oak Avenue",
                "city": "San Francisco",
                "state": "CA",
                "zipcode": "94102",
                "country": "USA"
            }
        },
        "items": [
            {
                "item_id": "ITM-003",
                "product_name": "Smart Watch",
                "category": "Electronics",
                "quantity": 1,
                "unit_price": 299.99,
                "discount_percent": 15,
                "offers": [
                    {"offer_id": "OFF-003", "offer_type": "FLASH_SALE", "discount_value": 20.0}
                ]
            }
        ],
        "payment": {
            "payment_id": "PAY-5002",
            "payment_mode": "PayPal",
            "payment_status": "SUCCESS",
            "transaction_id": "TXN-98766"
        },
        "order_status": "SHIPPED"
    },
    {
        "order_id": "ORD-2026-003",
        "order_date": "2026-05-17",
        "order_timestamp": "2026-05-17T09:15:30Z",
        "customer": {
            "customer_id": "CUST-1003",
            "customer_name": "Mike Johnson",
            "email": "mike.j@email.com",
            "phone": None,  # Missing phone number
            "address": {
                "street": "789 Pine Road",
                "city": "Austin",
                "state": "TX",
                "zipcode": "73301",
                "country": "USA"
            }
        },
        "items": [
            {
                "item_id": "ITM-004",
                "product_name": "Laptop Stand",
                "category": "Office Supplies",
                "quantity": 2,
                "unit_price": 45.00,
                "discount_percent": 5,
                "offers": None  # No offers available
            },
            {
                "item_id": "ITM-005",
                "product_name": "Mechanical Keyboard",
                "category": "Electronics",
                "quantity": 1,
                "unit_price": 129.99,
                "discount_percent": 0,
                "offers": [
                    {"offer_id": "OFF-004", "offer_type": "BUNDLE", "discount_value": 10.0}
                ]
            }
        ],
        "payment": {
            "payment_id": "PAY-5003",
            "payment_mode": "Debit Card",
            "payment_status": "PENDING",
            "transaction_id": "TXN-98767"
        },
        "order_status": "PROCESSING"
    }
]

# Create JSON file in Workspace (not DBFS since public DBFS is disabled)
import os

# Define workspace path
workspace_path = "/Workspace/Users/pavankumarreddy3247@gmail.com/Retail Sales/retail-medallion-project/nested_json_demo"

# Create directory if it doesn't exist
os.makedirs(workspace_path, exist_ok=True)

# Write JSON file (one JSON object per line - JSONL format)
file_path = f"{workspace_path}/orders.json"
with open(file_path, "w") as f:
    for order in sample_orders:
        f.write(json.dumps(order) + "\n")

print("✅ Sample nested JSON data created successfully!")
print(f"📁 Location: {file_path}")
print(f"📊 Records: {len(sample_orders)}")
print("\n📝 Sample Record:")
print(json.dumps(sample_orders[0], indent=2)[:800] + "...")

## Step 2: Define Explicit Schema using StructType

**Why explicit schema?**
- ✅ **Performance**: Avoids expensive schema inference on large datasets
- ✅ **Data Quality**: Enforces expected data types
- ✅ **Production Ready**: Schema evolution control
- ✅ **Interview Focus**: Demonstrates deep understanding of PySpark types

**Key Components**:
- `StructType`: Represents nested objects/structs
- `ArrayType`: Represents arrays/lists
- `StructField`: Individual field definition with nullable flag